In [50]:
!pip install openai-agents -q

### Setup and Imports

In [51]:
import os
from sys import exception
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("No Open AI Key")

MODEL = "gpt-4.1-mini"

In [52]:
WEB_AGENT_PROMPT = """ 
You are a helpful assistant that can read the web pages.
When the user asks about a web page , if you can access the internet, 
then read thatpage  first, then answer based on 
what you actually read. Always Cite the URL.

If you cann't access the internet, then say so, dont make things up.
"""

#If you cann't access the internet, then say so, dont make things up.

In [53]:
web_agent = Agent(
    name= "web agent",
    instructions=WEB_AGENT_PROMPT,
    model = MODEL
)

In [54]:
result = await Runner.run(
    web_agent,
    input = "who won the last game on IPL 2023",
    #input = "who won the last game on IPL 2023, read https://www.iplt20.com/",
    #input = "Is gaurav works in https://www.linkedin.com/in/gaurav-kamath-1b0a3b1a7/",
    max_turns=10
)

print(f"Last Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

Last Agent: web agent
-----


I currently do not have the ability to access real-time internet data to provide the latest match results. For the most recent IPL 2023 game winner, please check a reliable sports news website or the official IPL site. If you have any other questions or need information up to my last update, feel free to ask!

### Launch our first MCP Server

#### Step 1: Descrive Launch Parameters

In [43]:
fetch_server_params = {
    "command": "uvx",
    "args" : ["mcp-server-fetch"]
}

#### Lets investigate the Server

In [57]:
async with MCPServerStdio(name = "Fetch Server", params=fetch_server_params, 
                          client_session_timeout_seconds=60) as server1:
    tools = await server1.list_tools()
    print(f"✅ Connected. the server offer {len(tools)} tools: \n")
    for tool in tools:
        print(f"🛠️ {tool.name}")
        print(f"{tool.description}")


✅ Connected. the server offer 1 tools: 

🛠️ fetch
Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.


#### Step 2: Launch and Connect to the MCP sever

In [48]:
async with MCPServerStdio(name = "Fetch Server", params=fetch_server_params, 
                          client_session_timeout_seconds=60) as server:
    web_agent = Agent(
        name= "web agent",
        instructions=WEB_AGENT_PROMPT,
        model = MODEL,
        mcp_servers=[server]
    )

    result = await Runner.run(
        web_agent,
        input = "who won the last game on IPL",
        #input = "who won the last game on IPL, read https://www.iplt20.com/",
        #input = "Is gaurav works in https://www.linkedin.com/in/gaurav-kamath-1b0a3b1a7/",
        max_turns=10
    )

    print(f"Last Agent: {result.last_agent.name}")
    print("-----")
    display(Markdown(result.final_output))

Last Agent: web agent
-----


The last game in the IPL was the TATA IPL 2026 final, where Royal Challengers Bengaluru (RCB) won against Gujarat Titans (GT) by five wickets. RCB chased a target of 156 and reached 161/5 with 12 balls to spare, successfully defending their crown and securing their second IPL title. Virat Kohli played a key role in the chase, remaining unbeaten on 75 off 42 balls and was named Player of the Match.

You can find more details here: https://www.iplt20.com/news/article/tata-ipl-2026-final-rcb-v-gt-match-report